In [ ]:
!pip install pyvis spacy stanza networkx rapidfuzz flair
!python -m spacy download fr_core_news_lg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 13.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 44.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 MB 2.4 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [3]:
import sys
import os
PROJECT_PATH = "/content/drive/MyDrive/AMS3"
sys.path.append(PROJECT_PATH)

# Character Network Extraction - Optimized Workflow

**Workflow**:
1. **Setup**: Import libraries and configure paths
2. **NER Extraction** (slow, run once): Extract characters from all chapters → store in memory
3. **Co-occurrence + Submission** (fast, re-run as needed): Use cached NER results with different parameters

## 1. Setup & Imports

In [ ]:
import importlib
import networkx as nx
import pandas as pd
import time
from pathlib import Path

# Reload modules
importlib.reload(importlib.import_module("nlp_cooccurrence"))
importlib.reload(importlib.import_module("nlp_multi_ner"))
importlib.reload(importlib.import_module("nlp_extract_characters"))
importlib.reload(importlib.import_module("nlp_aliases"))
importlib.reload(importlib.import_module("nlp_graph"))

from nlp_multi_ner import ensemble_entities
from nlp_extract_characters import count_entities, filter_persons, filter_locations
from nlp_utils import read_file, load_anti_dict
from nlp_aliases import group_aliases, alias_dictionary, merge_alias_counts, filter_by_frequency
from nlp_cooccurrence import detect_cooccurrences
from nlp_graph import generate_graph, remove_isolated_nodes

print("✅ Modules loaded!")


✅ Modules loaded!


## 2. Configuration

In [5]:
# Data paths
books_folder = "/content/drive/MyDrive/AMS3/data"
books_config = [
    (list(range(0, 19)), "paf", f"{books_folder}/prelude_a_fondation"),
    (list(range(0, 18)), "lca", f"{books_folder}/les_cavernes_d_acier"),
]

# Anti-dictionary for filtering false positives
anti_dict_file = "/content/drive/MyDrive/AMS3/antidict.txt"
anti_dict = load_anti_dict(anti_dict_file)

print(f"📚 Books configured: {len(books_config)} books")
print(f"🚫 Anti-dictionary loaded: {len(anti_dict)} entries")

📚 Books configured: 2 books
🚫 Anti-dictionary loaded: 702 entries


## 3. NER Extraction (Run Once - Slow ~2-5 min)

Extract all characters from all chapters and store in memory. **Only re-run this if you change NER/filtering logic.**

In [6]:
ner_cache = {}  # {chapter_id: {'text': str, 'LP': Counter, 'LL': Counter}}

In [ ]:
# Storage for NER results (in memory)

print("="*60)
print("🧠 EXTRACTING CHARACTERS (NER + FILTERING)")
print("="*60)
print()

start_time = time.time()

for chapter_range, book_code, folder_path in books_config:
    for chapter_num in chapter_range:
        chapter_id = f"{book_code}{chapter_num}"
        chapter_file = f"{folder_path}/chapter_{chapter_num + 1}.txt.preprocessed"

        print(f"Processing {chapter_id}...", end=" ", flush=True)

        # Read text
        text = read_file(chapter_file)

        # Extract entities using ensemble NER
        raw_entities = ensemble_entities(text, method="vote")

        # Count entities
        L = count_entities(raw_entities)

        # Filter persons
        LP = filter_persons(L, anti_dict=anti_dict)

        # Filter locations (optional, can be used for graph context)
        LL = filter_locations(L)

        # Store in cache (no alias grouping yet)
        ner_cache[chapter_id] = {
            'text': text,
            'L': L,
            'LP': LP,  # Raw person counts (before alias merging)
            'LL': LL
        }

        print(f"✓ {len(LP)} raw entities")

elapsed = time.time() - start_time
print()
print("="*60)
print(f"✅ NER EXTRACTION COMPLETE")
print(f"⏱️  Time: {elapsed:.2f}s ({elapsed/60:.2f} min)")
print(f"📊 Cached: {len(ner_cache)} chapters")
print("="*60)

🧠 EXTRACTING CHARACTERS (NER + FILTERING)

Processing paf0... ✓ 9 raw entities
Processing paf1... ✓ 10 raw entities
Processing paf2... ✓ 6 raw entities
Processing paf3... ✓ 14 raw entities
Processing paf4... ✓ 13 raw entities
Processing paf5... ✓ 11 raw entities
Processing paf6... ✓ 15 raw entities
Processing paf7... ✓ 15 raw entities
Processing paf8... ✓ 13 raw entities
Processing paf9... ✓ 8 raw entities
Processing paf10... ✓ 17 raw entities
Processing paf11... ✓ 16 raw entities
Processing paf12... ✓ 27 raw entities
Processing paf13... ✓ 21 raw entities
Processing paf14... ✓ 20 raw entities
Processing paf15... ✓ 19 raw entities
Processing paf16... ✓ 26 raw entities
Processing paf17... ✓ 23 raw entities
Processing paf18... ✓ 16 raw entities
Processing lca0... ✓ 12 raw entities
Processing lca1... ✓ 10 raw entities
Processing lca2... ✓ 7 raw entities
Processing lca3... ✓ 20 raw entities
Processing lca4... ✓ 15 raw entities
Processing lca5... ✓ 10 raw entities
Processing lca6... ✓ 14 raw

## 3.5. Save/Load NER Cache (Optional - Persist Results)

**Save** NER results to disk so they survive kernel restarts. **Load** to skip NER extraction entirely.

In [7]:
import pickle

# Cache file path - save in project directory (Google Drive)
project_dir = Path("/content/drive/MyDrive/AMS3")
cache_file = project_dir / "ner_cache.pkl"

# =============================================================================
# SAVE NER CACHE TO FILE
# =============================================================================
def save_ner_cache():
    """Save the ner_cache dictionary to disk (Google Drive)"""
    if not ner_cache:
        print("⚠️  NER cache is empty. Run cell 3 first.")
        return

    with open(cache_file, 'wb') as f:
        pickle.dump(ner_cache, f)

    file_size = cache_file.stat().st_size / (1024 * 1024)  # MB
    print(f"✅ Saved {len(ner_cache)} chapters to Google Drive")
    print(f"   Location: {cache_file.name}")
    print(f"   File size: {file_size:.2f} MB")

# =============================================================================
# LOAD NER CACHE FROM FILE
# =============================================================================
def load_ner_cache():
    """Load the ner_cache dictionary from disk (Google Drive)"""
    global ner_cache

    if not cache_file.exists():
        print(f"❌ Cache file not found in Google Drive")
        print(f"   Expected: {cache_file.name}")
        print("   Run cell 3 to extract NER, then save with save_ner_cache()")
        return False

    with open(cache_file, 'rb') as f:
        ner_cache = pickle.load(f)

    file_size = cache_file.stat().st_size / (1024 * 1024)  # MB
    print(f"✅ Loaded {len(ner_cache)} chapters from Google Drive")
    print(f"   Location: {cache_file.name}")
    print(f"   File size: {file_size:.2f} MB")
    print(f"   You can now skip cell 3 and go directly to cell 4!")
    return True

# =============================================================================
# AUTO-SAVE AFTER NER EXTRACTION
# =============================================================================
def auto_save_after_ner():
    """Automatically save cache after running NER extraction"""
    if ner_cache and len(ner_cache) > 0:
        save_ner_cache()
        print("   💾 Auto-saved to Google Drive")

# =============================================================================
# USAGE
# =============================================================================
print("📁 NER Cache Management (Google Drive)")
print("-" * 60)
print(f"Cache location: {cache_file}")
print()
print("Commands available:")
print("  • save_ner_cache()       - Save cache to Google Drive")
print("  • load_ner_cache()       - Load cache from Google Drive")
print("  • auto_save_after_ner()  - Auto-save after cell 3")
print()
print("Workflow:")
print("  1. Run cell 3 (NER extraction)")
print("  2. Cache is automatically in Google Drive sync folder")
print("  3. Next session: Run load_ner_cache() to skip cell 3")
print("-" * 60)

# Auto-load if cache exists and ner_cache is empty
if cache_file.exists() and not ner_cache:
    print()
    print("🔍 Found existing cache in Google Drive!")
    print("   Run: load_ner_cache()")
elif ner_cache:
    print()
    print(f"✓ NER cache in memory: {len(ner_cache)} chapters")
    print("  To save to Google Drive: run save_ner_cache()")

📁 NER Cache Management (Google Drive)
------------------------------------------------------------
Cache location: /content/drive/MyDrive/AMS3/ner_cache.pkl

Commands available:
  • save_ner_cache()       - Save cache to Google Drive
  • load_ner_cache()       - Load cache from Google Drive
  • auto_save_after_ner()  - Auto-save after cell 3

Workflow:
  1. Run cell 3 (NER extraction)
  2. Cache is automatically in Google Drive sync folder
  3. Next session: Run load_ner_cache() to skip cell 3
------------------------------------------------------------

🔍 Found existing cache in Google Drive!
   Run: load_ner_cache()


In [ ]:
save_ner_cache()

✅ Saved 37 chapters to Google Drive
   Location: ner_cache.pkl
   File size: 0.74 MB


In [8]:
load_ner_cache()

✅ Loaded 37 chapters from Google Drive
   Location: ner_cache.pkl
   File size: 0.74 MB
   You can now skip cell 3 and go directly to cell 4!


True

## 4. Co-occurrence Detection + Submission (Fast - Re-run as needed)

**Change parameters below and re-run this cell** to test different settings without re-running NER.

- `distance_max` — co-occurrence window size in words
- `min_occurrences` — minimum number of mentions to keep a character (applied after alias merging)

In [9]:
def read_file(path):
    with open(path, "r", encoding="utf8") as f:
        return f.read()

def load_anti_dict(path="antidict.txt"):
    s = set()
    try:
        with open(path, "r", encoding="utf8") as f:
            for line in f:
                # remove inline comments and surrounding whitespace
                content = line.split("#", 1)[0].strip()
                if not content:
                    continue
                s.add(content.lower())
        return s
    except FileNotFoundError:
        return set()

In [ ]:
# ============================================================
# ADJUST THESE PARAMETERS AND RE-RUN THIS CELL
# ============================================================
distance_max = 150      # Co-occurrence window size in words
min_occurrences = 2     # Minimum mentions to keep a character (after alias merging)
# ============================================================

output_csv = f"submission_{distance_max}_min{min_occurrences}.csv"

books_folder = "/content/drive/MyDrive/AMS3/data"
books_config = [
    # (chapter_range, book_code, folder_path)
    (list(range(0, 19)), "paf", f"{books_folder}/prelude_a_fondation"),      # paf0 to paf18
    (list(range(0, 18)), "lca", f"{books_folder}/les_cavernes_d_acier"),     # lca0 to lca17
]

print("="*60)
print(f"🔗 DETECTING CO-OCCURRENCES (window={distance_max}, min_occ={min_occurrences})")
print("="*60)
print()

if not ner_cache:
    print("❌ ERROR: NER cache is empty! Run the NER extraction cell first.")
else:
    start_time = time.time()

    submission_data = []

    for chapters, book_code, folder_path in books_config:
      print(f"\n📖 Processing book: {book_code} ({len(chapters)} chapters)")
      for chapter_num in chapters:
            chapter_id = f"{book_code}{chapter_num}"

            chapter_file = os.path.join(
                folder_path,
                f"chapter_{chapter_num + 1}.txt.preprocessed"
            )

            text = read_file(chapter_file)

            # Check if file exists
            if not os.path.isfile(chapter_file):
                print(f"   ⚠️  {chapter_id}: File not found: {chapter_file}")
                continue

            try:
                print(f"   📄 Processing {chapter_id}...", end=" ")

                LP = ner_cache[chapter_id]['LP']

                groups = group_aliases(LP)
                alias_map = alias_dictionary(groups)
                LP_merged = merge_alias_counts(LP, alias_map)

                # Filter low-frequency characters (after alias merging so aliases are summed)
                LP_before = len(LP_merged)
                # LP_merged = filter_by_frequency(LP_merged, min_occurrences)
                LP_after = len(LP_merged)

                cooccurrences = detect_cooccurrences(text, LP_merged, distance_max)

                G = generate_graph(cooccurrences, LP_merged)

                # Remove characters that never co-appear with anyone
                nodes_before = G.number_of_nodes()
                remove_isolated_nodes(G)
                nodes_removed = nodes_before - G.number_of_nodes()

                for group in groups:
                  canonical = group[0]  # First name is canonical
                  if canonical in G.nodes:
                      # Join all aliases with semicolon as required
                      G.nodes[canonical]["names"] = ";".join(group)

                # Handle nodes that might not be in groups (single occurrence names)
                for node in G.nodes:
                    if "names" not in G.nodes[node]:
                        G.nodes[node]["names"] = node

                # Convert to GraphML string
                graphml_str = "".join(nx.generate_graphml(G))

                # Add to submission
                submission_data.append({
                  'ID': chapter_id,
                  'graphml': graphml_str
                })

                filtered_msg = f", filtered {LP_before}→{LP_after} chars" if LP_before != LP_after else ""
                isolated_msg = f", removed {nodes_removed} isolated" if nodes_removed > 0 else ""
                print(f"✓ (Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}{filtered_msg}{isolated_msg})")

            except Exception as e:
                print(f"❌ Error: {e}")
                continue

    # Create DataFrame and save
    df_submission = pd.DataFrame(submission_data)
    df_submission.set_index('ID', inplace=True)
    df_submission.to_csv(output_csv)

    elapsed = time.time() - start_time
    print()
    print("="*60)
    print(f"✅ SUBMISSION GENERATED")
    print(f"⏱️  Time: {elapsed:.2f}s")
    print(f"📄 File: {output_csv}")
    print(f"📊 Entries: {len(df_submission)}")
    print("="*60)


🔗 DETECTING CO-OCCURRENCES (window=150)


📖 Processing book: paf (19 chapters)
   📄 Processing paf0... ✓ (Nodes: 7, Edges: 14)
   📄 Processing paf1... ✓ (Nodes: 9, Edges: 26)
   📄 Processing paf2... ✓ (Nodes: 4, Edges: 6)
   📄 Processing paf3... ✓ (Nodes: 11, Edges: 30)
   📄 Processing paf4... ✓ (Nodes: 7, Edges: 10)
   📄 Processing paf5... ✓ (Nodes: 7, Edges: 21)
   📄 Processing paf6... ✓ (Nodes: 12, Edges: 43)
   📄 Processing paf7... ✓ (Nodes: 11, Edges: 19)
   📄 Processing paf8... ✓ (Nodes: 10, Edges: 21)
   📄 Processing paf9... ✓ (Nodes: 5, Edges: 5)
   📄 Processing paf10... ✓ (Nodes: 14, Edges: 46)
   📄 Processing paf11... ✓ (Nodes: 10, Edges: 17)
   📄 Processing paf12... ✓ (Nodes: 20, Edges: 82)
   📄 Processing paf13... ✓ (Nodes: 15, Edges: 36)
   📄 Processing paf14... ✓ (Nodes: 13, Edges: 34)
   📄 Processing paf15... ✓ (Nodes: 14, Edges: 47)
   📄 Processing paf16... ✓ (Nodes: 20, Edges: 102)
   📄 Processing paf17... ✓ (Nodes: 15, Edges: 62)
   📄 Processing paf18... ✓ (Nodes: 13,

## 5. Verification (Optional)

In [11]:
# Quick verification of submission
print("📊 Submission Overview:")
print(df_submission.head(10))
print()

# Sample graph inspection
sample_id = "paf0"
if sample_id in df_submission.index:
    import io
    graphml_str = df_submission.loc[sample_id, "graphml"]
    G_sample = nx.read_graphml(io.StringIO(graphml_str))

    print(f"Sample: {sample_id}")
    print(f"  Nodes: {G_sample.number_of_nodes()}")
    print(f"  Edges: {G_sample.number_of_edges()}")
    print(f"\n  Top 5 nodes:")
    for i, (node, data) in enumerate(list(G_sample.nodes(data=True))[:5]):
        names = data.get('names', '⚠️ MISSING')
        print(f"    • {node}: {names}")

📊 Submission Overview:
                                                graphml
ID                                                     
paf0  <graphml xmlns="http://graphml.graphdrawing.or...
paf1  <graphml xmlns="http://graphml.graphdrawing.or...
paf2  <graphml xmlns="http://graphml.graphdrawing.or...
paf3  <graphml xmlns="http://graphml.graphdrawing.or...
paf4  <graphml xmlns="http://graphml.graphdrawing.or...
paf5  <graphml xmlns="http://graphml.graphdrawing.or...
paf6  <graphml xmlns="http://graphml.graphdrawing.or...
paf7  <graphml xmlns="http://graphml.graphdrawing.or...
paf8  <graphml xmlns="http://graphml.graphdrawing.or...
paf9  <graphml xmlns="http://graphml.graphdrawing.or...

Sample: paf0
  Nodes: 7
  Edges: 14

  Top 5 nodes:
    • Empereur: Empereur
    • Hari Seldon: Hari Seldon;Seldon
    • Cléon: Cléon
    • Eto Demerzel: Eto Demerzel;Demerzel
    • Sire: Sire


## 6. Test Multiple Window Sizes (Optional)

Run this to compare multiple `distance_max` values quickly using cached NER results.

In [ ]:
# Test multiple window sizes
window_sizes = [25, 50, 75, 100, 150]

results_comparison = {}

for distance in window_sizes:
    print(f"\n🔍 Testing window size: {distance}")

    start = time.time()

    total_nodes = 0
    total_edges = 0

    for chapter_id, cached_data in ner_cache.items():
        text = cached_data['text']
        LP_merged = cached_data['LP_merged']

        cooccurrences = detect_cooccurrences(text, LP_merged, distance)
        G = generate_graph(cooccurrences, LP_merged)

        total_nodes += G.number_of_nodes()
        total_edges += G.number_of_edges()

    elapsed = time.time() - start
    avg_nodes = total_nodes / len(ner_cache)
    avg_edges = total_edges / len(ner_cache)

    results_comparison[distance] = {
        'avg_nodes': round(avg_nodes, 2),
        'avg_edges': round(avg_edges, 2),
        'time_seconds': round(elapsed, 2)
    }

    print(f"  ⏱️  {elapsed:.2f}s | Avg: {avg_nodes:.1f} nodes, {avg_edges:.1f} edges")

# Summary
print("\n" + "="*60)
print("📊 WINDOW SIZE COMPARISON")
print("="*60)
results_df = pd.DataFrame(results_comparison).T
results_df.index.name = 'window_size'
print(results_df)